# Proyecto Start-up: Guía turística inteligente
Asignatura: Aprendizaje profundo (Máster Ciencia de Datos)

Autoras: Lucía Benages Guijarro e Irene Barba La Orden

## 0. Setup

In [3]:
# Librerías necesarias

# ! pip install SpeechRecognition pydub
# ! pip install langdetect
# ! pip install openai-whisper
# ! pip install torch
# ! pip install pyaudio
# #! pip install ollama
# ! pip install gtts
# ! pip install playsound==1.2.2
# ! pip install edge_tts
# !pip install deep-translator
# !pip install easyocr
# !pip install transformers

import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect
import re
from pydub import AudioSegment
import os
from gtts import gTTS
from playsound import playsound
import edge_tts
import asyncio
import json
import tkinter as tk
from tkinter import filedialog
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, AutoTokenizer
from PIL import Image
import os.path
import torch
import pandas as pd
from deep_translator import GoogleTranslator
import easyocr
import cv2
import subprocess
from IPython.display import clear_output
import warnings
from google.colab import files
from IPython.display import Audio, display
import copy

In [4]:
# En colab utilizaremos un único modelo (qwen3) para tanto para texto
# como identificación de imagen
MODELO_CHAT = "Qwen/Qwen3-VL-4B-Instruct"

global model
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# Cargamos ya el tokenizer y preprocessor para que el código sea más rápido
tokenizer = AutoTokenizer.from_pretrained(MODELO_CHAT)
processor = AutoProcessor.from_pretrained(MODELO_CHAT)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

In [5]:
# Funciones específicas que necesitamos para trabajar en colab
def respuesta_modelo_t2t(prompt, max_tokens):
  """
  Generación de la respuesta del modelo cuando se quiere utilizar de texto
  a texto.
  Argumentos:
  - prompt: input de texto para el modelo.
  - max_tokens: número máximo de tokens para la respuesta generada.
  """
  device = model.device
  messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]
  text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  input_text = processor(
        text=[text],
        padding=True,
        return_tensors="pt"
    )
  input_text = {k: v.to(device) for k, v in input_text.items()}

  # Generamos la respuesta
  outputs = model.generate(**input_text, max_new_tokens=max_tokens)
  outputs_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(input_text["input_ids"], outputs)
    ]
  # Se decodifica
  respuesta_decodificada = processor.batch_decode(
        outputs_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
  respuesta_ia = respuesta_decodificada[0].strip()

  return respuesta_ia

def respuesta_modelo_i2t2t(imagen, pregunta_texto):
  """
  Generación de la respuesta del modelo cuando se quiere utilizar de
  imagen-texto a texto.
  Argumentos:
  - imagen: input tipo imagen del modelo
  - pregunta_texto: input de texto para el modelo, cuestión a responder.
  """

  # Preprocesado de la entrada
  processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")
  messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": imagen},
                {"type": "text", "text": pregunta_texto},
            ],
        }
    ]

  text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = processor(
        text=[text],
        images=[imagen],
        padding=True,
        return_tensors="pt"
    )

    # Generación de la respuesta
  generated_ids = model.generate(**inputs, max_new_tokens=50)

  generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]

  respuesta_decodificada = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
  frase_original = respuesta_decodificada[0].strip()

  return frase_original


In [6]:
# Para esconder mensajes de carga durante la conversación de la app
from transformers import logging
logging.set_verbosity_error()
logging.disable_progress_bar()
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Input

In [7]:
def audio_a_wav(audio):
    """
    Esta función devuelve un archivo de audio en formato .wav.
    Argurmento:
    - audio: archivo de audio en cualquier formato.
    """
    nombre, extension = os.path.splitext(audio)
    extension = extension.lower()

    if extension == ".wav":
        return audio
    else:
        print("Convirtiendo el archivo a .wav.")
        archivo_nuevo = nombre + "convertido.wav"
        try:
            audio = AudioSegment.from_file(audio)
            audio.export(archivo_nuevo, format="wav")
            print("Archivo convertido a .wav.")
            return archivo_nuevo
        except:
            print("Error al convertir el archivo a .wav.")
            return None

In [8]:
device = "mps" if torch.backends.mps.is_available() else "cpu" # (para MAC)
# Modelo base (whisper)
sound_model = w.load_model("base", device=device)

def input_speech(audio = None):
    """
    Esta función transcribe un audio en formato .wav a texto o activa un micrófono en directo para guardar temporalmente el audio. Después de obtener el audio, lo transcribe y devuelve la transcripción y el idioma.
    speech_recognition: detecta el ruido de fondo, detección de voz y detector de silencios
    whisper: identifica el idioma, transcribe el audio.
    Argumento:
    - audio: archivo de audio en formatio .wav (por defecto, audio = None).
    """
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        # Pasar a formato .wav
        audio_a_wav(audio)
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente y requiere internet)
        result = sound_model.transcribe(audio)
        # Se devuelve la transcripción del audio y el idioma
        return result["text"].strip(), result["language"]

    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio
            r.adjust_for_ambient_noise(source, duration=1)
            # Aunque el usuario se queda callado 2 segundos, no se corta la grabación
            r.pause_threshold = 2.0
            # Se escucha el audio durante 20 segundos
            audio_data = r.listen(source, phrase_time_limit=20)

            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())

            result = sound_model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma (aunque al principio se indique en qué idioma se quiere la app, detectamos el idioma de la persona al hablar)
            return result["text"].strip(), result["language"]

### 1.1. Texto (text to text)

In [9]:
def input_text():
    """
    Esta función pide un texto al usuario, detectando el idioma. Devuelve el texto escrito y el idioma.
    """
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma (aunque al principio se indique el idioma de la app)
    try:
        # De la librería langdetect (no necesita internet y detecta 55 idiomas)
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"

    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

### 1.2. Cleaning text

In [10]:
def cleaning_text(texto, idioma):
    """
    Esta función limpia un texto, para mejorar la legibilidad para después pasárselo a la IA. Devuelve el texto corregido.
    Argumentos:
    - texto: texto a limpiar.
    - idioma: idioma en el que está el texto.
    """
    # strip para eliminar espacios en blanco
    # capitalize para poner la primera letra en mayúscula
    texto_limpio = texto.strip().capitalize()

    # Muletillas a eliminar (español e inglés)
    muletillas = {
        "es": [r"\btipo+\b", r"\beh+\b", r"\besto+\b", r"\bpues+\b", r"\ben\s+plan\b"],
        "en": [r"\bum+\b", r"\beh+\b", r"\blike\b", r"\byou know\b"]
    }

    # Eliminamos las muletillas
    if idioma in muletillas:
        for i in muletillas[idioma]:
            texto_limpio = re.sub(i, "", texto_limpio, flags = re.IGNORECASE)

    # Eliminamos los n espacios que se han creado al eliminar las muletillas y lo sustitumos por un espacio
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

    return texto_limpio

### 1.3. Leer y traducir texto de una imagen

In [11]:
def leer_imagen(ruta_imagen, idioma=['en']):
  """
  Función que extrae texto de una imagen.
  Argumentos:
  - ruta_imagen: nombre del archivo de imagen a procesar.
  - idioma: código ISO del idioma que se debe reconocer.
  """
  # Cargar imagen
  image = cv2.imread(ruta_imagen)

  # Cargar lector
  reader = easyocr.Reader(idioma, verbose=False)

  # Leer texto
  lectura = reader.readtext(image, paragraph=True, detail=0)
  return lectura


def menu_idiomas(texto_opciones, texto_despedidas, texto_traduciendo, idioma):
  """
  Función que carga una imagen, procesa la imagen, extrae el texto para traducirlo al idioma deseado, devolviéndolo.
  Argumentos:
  - texto_opciones: menú de los alfabetos disponibles.
  - texto_despedidas: mensaje que se devuelve si el usuario indica la opción 7.
  - texto_traduciendo: mensaje que se muestra mientras se procesa la traducción.
  - idioma: código ISO del idioma al que se debe traducir el texto extraído.
  """

  # Encontramos la ruta de la imagen dejando que el usuario la suba
  imagen_subida = files.upload()
  ruta = list(imagen_subida.keys())[0]

  print(texto_opciones, flush=True)

  opcion = "aaaa"
  while opcion not in ["1","2","3","4","5","6","7"]:
    opcion = input("\nSelecciona una opción (1, 2, 3, 4, 5, 6 o 7): ")
    if opcion == "1":
      texto = leer_imagen(ruta, ['en'])
    if opcion == "2":
      texto = leer_imagen(ruta, ['ru','en'])
    if opcion == "3":
      texto = leer_imagen(ruta, ['ch_tra','en'])
    if opcion == "4":
      texto = leer_imagen(ruta, ['ch_sim', 'en'])
    if opcion == "5":
      texto = leer_imagen(ruta, ['ja', 'en'])
    if opcion == "6":
      texto = leer_imagen(ruta, ['ko','en'])
    if opcion == "7":
      return texto_despedidas

  texto_junto = " \n".join(texto)
  # Limpiar el texto a traducir
  texto_junto = re.sub('[$><|+`~]', '', texto_junto)
  print(texto_traduciendo)
  texto_traducido = GoogleTranslator(source='auto', target=idioma).translate(texto_junto)

  return texto_traducido

### 1.4. Identificación de un monumento con una foto

In [12]:
def info_con_imagen(idioma, lugar_viaje):
    """
    Identifica un monumento a partir de una imagen usando Qwen3-VL.
    Devuelve una frase descriptiva en el idioma seleccionado.
    Argumentos:
    - idioma: código ISO del idioma en el que se debe responder.
    - lugar_viaje: lugar geográfico para acotar la búsqueda del monumento.
    """
    device = "mps" if torch.backends.mps.is_available() else "cpu"

    # Encontramos la ruta de la imagen
    imagen_subida = files.upload()
    ruta = list(imagen_subida.keys())[0]

    if not ruta:
        return "No se seleccionó ninguna imagen."

    # Se carga la imagen
    imagen = Image.open(BytesIO(imagen_subida[ruta])).convert("RGB")

    pregunta_texto = (
        f"You MUST NECESSARILY answer in the language with the ISO code: {idioma}."
        f"Identify the cultural or historical site in this image located in {lugar_viaje.upper()}. "
        f"Answer only with the name of the site and its location, in the language with the ISO code: {idioma}. Example: 'Sydney Opera, Australia.'"
    )

    frase_original = respuesta_modelo_i2t2t(imagen, pregunta_texto)

    try:
        frase_identificacion = GoogleTranslator(source='auto', target=idioma).translate(frase_original)
    except Exception:
        # Si falla el traductor, devolvemos la original para no romper la app
        frase_identificacion = frase_original


    print(f"\n{frase_identificacion}")

    # Devolvemos la frase para que la función app() la use en el siguiente paso
    return frase_identificacion


## 2. Generación de la respuesta (text to text)

In [13]:
def respuesta_texto(tema_o_monumento, idioma, opciones, lugar, es_monumento=False):
    """
    Función única para generar respuestas de la IA.
    Argumentos:
    - tema_o_monumento: texto a explicar (o el nombre detectado por Qwen).
    - idioma: código ISO del idioma.
    - opciones: diccionario de estilos de la interfaz.
    - lugar: lugar del viaje (contexto).
    - es_monumento: booleano. Si es True, aplica reglas estrictas de la UNESCO.
    """
    if es_monumento:
        estilo_instruccion = "Provide a professional historical introduction based on UNESCO facts. Include the country."
    else:
        print("\n========================================================")
        print(f"{opciones['titulo']}")
        print("========================================================")
        print(f"{opciones['pregunta']}")
        print(opciones['opcion_1'])
        print(opciones['opcion_2'])
        print(opciones['opcion_3'])

        op = input(f"\n")
        estilo_instruccion = opciones['instrucciones_ia'].get(op, opciones['instrucciones_ia']["1"])

    prompt = f"""
    You are an expert tour guide and historian.
    Your goal is to explain: {tema_o_monumento}.
    Context: The trip is in {lugar}.

    Specific Task: {estilo_instruccion}

    CRITICAL RULES:
    - If explaining a monument, use ONLY factual historical data. No hallucinations.
    - The response will be read aloud (TTS). DO NOT use emojis, symbols, or special characters.
    - Provide ONLY the spoken text.
    - MANDATORY: You must respond in the language of ISO code: {idioma}.
    """

    respuesta_ia = respuesta_modelo_t2t(prompt, 500)

    return respuesta_ia

## 4. Output (audio o texto)

In [19]:
async def respuesta_audio(respuesta_ia, idioma):
    """
    Esta función devuelve una respuesta por voz en un idioma dado (la respuesta es la que ha dado la IA, la pasamos a audio).
    Argumentos:
    - respuesta_ia: texto que ha dado la IA.
    - idioma: idioma en qué queremos la voz.
    """
    # Es mas rápido que por ejemplo Suno Bark (es más para reproducir una voz natural, no tan profesional) aunque requiere internet
    voces = await edge_tts.list_voices()
    voz_encontrada = None

    # Por defecto escogemos que el idioma "es" es "es-ES"
    if idioma == "es":
        for voz in voces:
            if "es-ES" in voz["ShortName"]:
                voz_encontrada = voz["ShortName"]
                break

    # Encontramos el idioma en el que queremos la respuesta
    if not voz_encontrada:
        for v in voces:
            if v["ShortName"].startswith(idioma):
                voz_encontrada = v["ShortName"]
                break

    # Si no encuentra la voz sobre ese idioma, por defecto usamos este
    if not voz_encontrada:
        voz_encontrada = "en-US-GuyNeural"

    # Guardamos temporalmente el audio
    archivo = "respuesta_voz.mp3"
    communicate = edge_tts.Communicate(respuesta_ia, voz_encontrada)
    await communicate.save(archivo)

    display(Audio(archivo))

In [15]:
async def menu_principal():
    """
    Función para determinar en qué idioma quiere que esté la app el usuario (se lo pasamos a la IA para que lo traduzca).
    Se puede decir el idioma por voz o escribiéndolo.
    Se devuelve el código ISO del idioma.
    """
    # Qué idioma quiere el usuario?
    print("========================================================")
    print("Configuración del idioma")
    print("========================================================")
    print("Escribe el idioma (ej: 'español', 'english', 'italiano').")

    op_idioma = input("\n")


    # También puede escribir el idioma que quiere
    prompt_iso = f"Provide ONLY the ISO 639-1 code (2 letters) for the language: {op_idioma}. Example: es, en, fr. Do not write anything else."
    cod_idioma = respuesta_modelo_t2t(prompt_iso, 10)

    # Si devuelve más de 2 letras.
    if len(cod_idioma) != 2:
        print(f"Aviso: No se reconoció '{op_idioma}'. Usando 'es' por defecto.")
        cod_idioma = "es"
    print(f"Idioma detectado: {cod_idioma}.")

    # Devolvemos el código del idioma
    return cod_idioma

In [16]:
# Menú a traducir según el idioma deseado por el usuario
MENU_COMPLETO = {
    "menu": {
        "bienvenida": "Menú principal",
        "opciones": [
            "1. Escribir una pregunta (Texto)",
            "2. Hacer foto a un monumento",
            "3. Traducir texto de una foto",
            "4. General historial del viaje",
            "5. Salir del menú"
        ],
        "instruccion": "Selecciona una opción (1, 2, 3, 4 o 5):",
    },
    "mensajes": {
        "pregunta_texto": "A continuación escriba su duda o el tema sobre el que desea información.",
        "pregunta_audio_modo": "¿Quiere (a) hablar o (b) subir un archivo?",
        "instruccion_audio": "Selecciona una opción (a o b):",
        "procesando": "Procesando información. Espere unos segundos.",
        "foto_monumento": "Este servicio permite identificar lugares de interés cultural o histórico a partir de una imagen.",
        "foto": "Sube una imagen.",
        "mas_info_foto": "¿Quieres más información sobre este monumento? Pulse (a) para seguir y cualquier otra tecla para salir.",
        "traduciendo": "Traduciendo texto...",
        "error_opcion": "Opción no válida. Inténtelo de nuevo.",
        "despedida": "¡Adiós!",
        "pregunta_escuchar": "¿Desea escuchar la respuesta? Pulse (a) para escuchar y cualquier otra letra para salir.",
        "enter": "Pulse ENTER para continuar.",
        "respuesta": "Respuesta:",
        "espera": "Generando audio. Espere unos segundos.",
        "historial_vacio": "No hay consultas guardadas todavía.",
        "historial_listo": "Historial generado. Busca {} en tu carpeta.",
        "aviso_foto": "AVISO: Intente realizar la foto lo más centrada posible al texto y asegúrese de que el texto sea lo principal en la imagen.",
        "confirmacion_foto": "¿Desea continuar? Pulse (a) para continuar o cualquier otra letra para volver al menú."

    },
    "estilos": {
        "titulo": "Opciones de estilo",
        "opcion_1": "1. Resumen breve",
        "opcion_2": "2. Historia detallada",
        "opcion_3": "3. Datos curiosos",
        "pregunta": "Selecciona el tipo de información deseada (1, 2 o 3):",
        "instrucciones_ia": {
            "1": "Hazme un resumen breve.",
            "2": "Hazme una explicación detallada y técnica.",
            "3": "Cuéntame curiosidades."
        }
    },
    "contexto": {
        "lugar": "¿A dónde es tu viaje?"
    },
    "opciones_alfabeto":
        """¿Qué alfabeto utiliza el texto a traducir?
            1. Latino
            2. Cirílico
            3. Chino tradicional
            4. Chino simplificado
            5. Japonés
            6. Coreano
            7. Salir
            """
}

async def menu_traducido(cod_idioma):
    """
    Función para traduciar el menú de la app en el idioma deseado.
    Argumento:
    - cod_idioma: código ISO del idioma al que se quiere traduciar la app.
    """
    if cod_idioma == 'es':
        return MENU_COMPLETO

    nuevo_menu = copy.deepcopy(MENU_COMPLETO)
    translator = GoogleTranslator(source='es', target=cod_idioma)

    nuevo_menu['menu']['bienvenida'] = translator.translate(nuevo_menu['menu']['bienvenida'])
    nuevo_menu['menu']['instruccion'] = translator.translate(nuevo_menu['menu']['instruccion'])
    nuevo_menu['menu']['opciones'] = [translator.translate(opt) for opt in nuevo_menu['menu']['opciones']]

    for clave, valor in nuevo_menu['mensajes'].items():
        nuevo_menu['mensajes'][clave] = translator.translate(valor)

    nuevo_menu['contexto']['lugar'] = translator.translate(nuevo_menu['contexto']['lugar'])

    nuevo_menu['estilos']['titulo'] = translator.translate(nuevo_menu['estilos']['titulo'])
    for i in ["opcion_1", "opcion_2", "opcion_3", "pregunta"]:
        nuevo_menu['estilos'][i] = translator.translate(nuevo_menu['estilos'][i])

    nuevo_menu['opciones_alfabeto'] = translator.translate(nuevo_menu['opciones_alfabeto'])

    return nuevo_menu


async def app():
    """
    Función ("app") que une todas la funciones y muestra el menú.
    """
    # Obtenemos el idioma deseado
    while True:
        print("========================================================")
        print("BIENVENIDO A TU GUÍA TURÍSTICA CON IA")
        print("========================================================")

        cod_idioma = await menu_principal()

        # Preguntamos si el idioma detectado es correcto
        print(f"\n¿Es correcto? Pulse (a) para SI o cualquier otra tecla para indicar de nuevo el idioma.")
        confirmacion = input(f"\n").lower().strip()

        if confirmacion == "a":
            print("Traduciendo al idioma deseado.")
            break

    # Traducimos la interfaz del menú
    interfaz = await menu_traducido(cod_idioma)

    print(f"\n{interfaz['contexto']['lugar']}")
    lugar_viaje = input(f"\n ")

    historial = []

    while True:
        #limpiar_pantalla()
        print("========================================================")
        print("BIENVENIDO A TU GUÍA TURÍSTICA CON IA")
        print("========================================================")

        print("========================================================")
        print(f"{interfaz['menu']['bienvenida']} ")
        print("========================================================")

        print(f"{interfaz['menu']['instruccion']}")

        for opt in interfaz['menu']['opciones']:
            print(opt)
        opcion = input(f"\n ")

        texto_sucio, idioma_final = None, cod_idioma

        # Opción para escribir
        if opcion == "1":
            print(f"\n{interfaz['mensajes']['pregunta_texto']}")
            texto_sucio = input("\n")

        # Hacer foto
        elif opcion == "2":
            print(f"\n{interfaz['mensajes']['foto_monumento']}")
            print(f"{interfaz['mensajes']['foto']}")

            frase_identificacion = info_con_imagen(idioma_final, lugar_viaje)

            print(f"\n{interfaz['mensajes']['mas_info_foto']}")
            sub_opcion = input("\n").lower().strip()

            if sub_opcion == "a":
                respuesta = respuesta_texto(
                    frase_identificacion,
                    idioma_final,
                    interfaz['estilos'],
                    lugar_viaje
                )
                historial.append({"pregunta": frase_identificacion, "respuesta": respuesta})
                print(f"\n{interfaz['mensajes']['pregunta_texto']}\n{respuesta}")

                print(f"\n{interfaz['mensajes']['pregunta_escuchar']}")
                audio_si_no = input("\n").lower().strip()

                if audio_si_no == "a":
                    print("Generando audio. Espere unos segundos...")
                    await respuesta_audio(respuesta, idioma_final)

                print(f"\n{interfaz['mensajes']['enter']}")
                input("\n")
            else:
                continue

        # Traducir texto de una imagen
        elif opcion == "3":
            print(interfaz['mensajes']['foto'])
            print(f"\n{interfaz['mensajes']['aviso_foto']}")
            print(f"\n{interfaz['mensajes']['confirmacion_foto']}")
            confirmacion_foto = input(f"\n")

            if confirmacion_foto == "a":
                traduccion = menu_idiomas(interfaz['opciones_alfabeto'],
                                interfaz['mensajes']['despedida'],
                                interfaz['mensajes']['traduciendo'],
                                idioma_final)
                historial.append({"pregunta": "Traducción de imagen", "respuesta": traduccion})
                print(traduccion)
                print(f"\n{interfaz['mensajes']['enter']}")
                input("\n")
            else:
                continue

        # Generar historial
        elif opcion == "4":
            if not historial:
                print(f"\n{interfaz['mensajes']['historial_vacio']}")
            else:
                # 1. Preparar la carpeta
                nombre_carpeta = "historiales"
                if not os.path.exists(nombre_carpeta):
                    os.makedirs(nombre_carpeta)

                # 2. Definir el nombre del archivo y la RUTA COMPLETA
                nombre_archivo = f"historial_{lugar_viaje.replace(' ', '_')}.txt"
                ruta_final = os.path.join(nombre_carpeta, nombre_archivo)

                # 3. GUARDAR usando la 'ruta_final' (esto es lo que faltaba)
                with open(ruta_final, "w", encoding="utf-8") as f:
                    f.write(f"--- HISTORIAL DE VIAJE: {lugar_viaje.upper()} ---\n\n")
                    for i, item in enumerate(historial, 1):
                        f.write(f"{i}. PREGUNTA: {item['pregunta']}\n")
                        f.write(f"   RESPUESTA: {item['respuesta']}\n")
                        f.write("-" * 50 + "\n\n")

                # 4. Avisar al usuario
                print(f"\n{interfaz['mensajes']['historial_listo'].format(ruta_final)}")
            print(f"{interfaz['mensajes']['enter']}")
            input(f"\n")
            continue

        # Salida
        elif opcion == "5":
            print(interfaz['mensajes']['despedida'])
            break

        # Error
        else:
            print(interfaz['mensajes']['error_opcion'])
            continue

        if texto_sucio:
            print(f"\n{interfaz['mensajes']['procesando']}\n")
            texto_limpio = cleaning_text(texto_sucio, idioma_final)
            respuesta = respuesta_texto(texto_limpio, idioma_final, interfaz['estilos'], lugar_viaje)
            historial.append({"pregunta": texto_limpio, "respuesta": respuesta})
            print(f"\n{interfaz['mensajes']['respuesta']}\n{respuesta}")

            print(f"\n{interfaz['mensajes']['pregunta_escuchar']}")
            audio_si_no = input("\n").lower().strip()

            if audio_si_no == "a":
                print(f"\n{interfaz['mensajes']['espera']}\n")
                await respuesta_audio(respuesta, idioma_final)

            print(f"\n{interfaz['mensajes']['enter']}")
            input("\n")


In [ ]:
await app()